# 🏥 Benchmark Evaluations Notebook 📈
  
**Goal**: Load ranking results for different retrieval methods, generate query/response pairs, compute per-query metrics (NDCG@10, NDCG@20, Recall@10), aggregate averages, select the best method, and upload to Azure AI Foundry.  
Phases:  
1. 📦 **Preprocess** (load data & build pairs)  
2. ⚙️ **Evaluate** (compute metrics)  
3. ➡️ **Post-process** (aggregate & upload)  

In [ ]:
import sys
import os
import json
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import evaluate
from azure.ai.evaluation._evaluate._eval_run import EvalRun

# ─── 0. Ensure project root is on PYTHONPATH ─────────────────────────────────
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.evals.custom.ndcg_evaluator import NDCGEvaluator
from src.evals.custom.search_recall_evaluator import SearchRecallEvaluator
import src.evals.sdk.custom_azure_ai_evaluations as custom_eval

# ─── 1. Load env & patch EvalRun ──────────────────────────────────────────────
load_dotenv()
EvalRun._start_run = custom_eval.custom_start_run

# ─── 2. Helper for tagging runs ───────────────────────────────────────────────
def _generate_custom_tags(case_id: str, git_commit: str, class_name: str):
    tags = [
        ("case", case_id),
        ("commit", git_commit),
        ("embeddings", "text-embedding-3-large"),
    ]
    custom_eval.CUSTOM_TAGS = tags
    return tags

# ─── 3. Configure AI Foundry client ───────────────────────────────────────────
conn_str = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")
_, AZ_SUBSCRIPTION_ID, AZ_RESOURCE_GROUP, AI_FOUNDRY_PROJECT_NAME = conn_str.split(";")
azure_ai_project = {
    "subscription_id": AZ_SUBSCRIPTION_ID,
    "resource_group_name": AZ_RESOURCE_GROUP,
    "project_name": AI_FOUNDRY_PROJECT_NAME,
}
credential = DefaultAzureCredential()
ai_project_client = AIProjectClient.from_connection_string(credential=credential, conn_str=conn_str)
print(f"Connected to Azure AI Project: {AI_FOUNDRY_PROJECT_NAME}")

# ─── 4. Git commit for tagging ─────────────────────────────────────────────────
git_hash = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"🔖 Using commit: {git_hash}")

# ─── 5. File paths ─────────────────────────────────────────────────────────────
EVAL_DIR = Path("../evals/benchmark/medindexer")
queries_path = EVAL_DIR / "queries.jsonl"
qrels_path   = EVAL_DIR / "qrels.jsonl"
ranking_files = {
    "hybrid_semantic": EVAL_DIR / "rankings-hybrid-semantic.jsonl",
    "hybrid":          EVAL_DIR / "rankings-hybrid.jsonl",
    "keyword":         EVAL_DIR / "rankings-keyword.jsonl",
    "vector":          EVAL_DIR / "rankings-vector.jsonl",
}

# ─── 6. Load queries & qrels ───────────────────────────────────────────────────
queries = {}
with open(queries_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        queries[e["id"]] = e["query"]

qrels = {}
with open(qrels_path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        qrels.setdefault(e["query"], {})[e["document"]] = int(e["relevant"])

# ─── 7. Build eval_data_<method>.jsonl ────────────────────────────────────────
for method, file_path in ranking_files.items():
    out_path = EVAL_DIR / f"eval_data_{method}.jsonl"
    with open(file_path, "r", encoding="utf-8") as fin, \
         open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)
            qid = rec["query"]
            # sort docs by score desc
            sorted_docs = [
                doc for doc, _ in sorted(rec["ranking"].items(), key=lambda x: x[1], reverse=True)
            ]
            record = {
                "query_id":     qid,
                "query":        queries.get(qid, ""),
                "method":       method,
                # embed the ranking under "context"
                "context":      { "ranking": rec["ranking"] },
                # emit ground_truth as a JSON object
                "ground_truth": qrels.get(qid, {}),
            }
            fout.write(json.dumps(record) + "\n")
    print(f"📦 Preprocessed data for '{method}' → {out_path}")

# ─── 8. Run evaluations ────────────────────────────────────────────────────────
for method in ranking_files:
    eval_file = EVAL_DIR / f"eval_data_{method}.jsonl"
    eval_name = f"search-retrieval-{method}-{git_hash}"

    _generate_custom_tags(case_id=method, git_commit=git_hash, class_name="SearchRetrievalEvaluator")

    result = evaluate(
        evaluation_name=eval_name,
        data=str(eval_file),
        evaluators={
            "ndcg_3":    NDCGEvaluator(3),
            "ndcg_10":   NDCGEvaluator(10),
            "recall_10": SearchRecallEvaluator(10),
        },
        azure_ai_project=azure_ai_project,
        evaluator_config={
            metric: {
                "column_mapping": {
                    # map the full context object
                    "context":      "${data.context}",
                    "ground_truth": "${data.ground_truth}",
                    "query_id":     "${data.query_id}"
                }
            }
            for metric in ["ndcg_3", "ndcg_10", "recall_10"]
        }
    )
    print(f"✅ Completed '{method}':", result["metrics"])

Connected to Azure AI Project: medevals
🔖 Using commit: a134c54
📦 Preprocessed data for 'hybrid_semantic' → ../evals/benchmark/medindexer/eval_data_hybrid_semantic.jsonl
📦 Preprocessed data for 'hybrid' → ../evals/benchmark/medindexer/eval_data_hybrid.jsonl
📦 Preprocessed data for 'keyword' → ../evals/benchmark/medindexer/eval_data_keyword.jsonl
📦 Preprocessed data for 'vector' → ../evals/benchmark/medindexer/eval_data_vector.jsonl


[2025-04-25 15:53:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:22 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_ndcg_evaluator_ndcgevaluator_3wlostko_20250425_155321_764385, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_3wlostko_20250425_155321_764385/logs.txt
[2025-04-25 15:53:22 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_

2025-04-25 15:53:22 -0700   35058 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-25 15:53:22 -0700   35058 execution.bulk     INFO     The timeout for the batch run is 3600 seconds.
2025-04-25 15:53:22 -0700   35058 execution.bulk     INFO     Current system's available memory is 8746.90625MB, memory consumption of current process is 330.0625MB, estimated available worker count is 8746.90625/330.0625 = 26
2025-04-25 15:53:22 -0700   35058 execution.bulk     INFO     Set process count to 4 by taking the minimum value among the factors of {'default_worker_count': 4, 'row_count': 35, 'estimated_worker_count_based_on_memory_usage': 26}.
2025-04-25 15:53:23 -0700   35058 execution.bulk     INFO     Process name(SpawnProcess-6)-Process id(35251)-Line number(0) start execution.
2025-04-25 15:53:23 -0700   35058 execution.bulk     INFO     Process name(SpawnProcess-11)-Process id(35261)-Line number(1) start execution.
202

[2025-04-25 15:53:38 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:39 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:39 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:39 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_ndcg_evaluator_ndcgevaluator_zs1l8n_u_20250425_155338_974086, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_zs1l8n_u_20250425_155338_974086/logs.txt
[2025-04-25 15:53:39 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_

======= Combined Run Summary (Per Evaluator) =======

{
    "ndcg_3": {
        "status": "Completed",
        "duration": "0:00:05.396278",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_yc5gbuvu_20250425_155321_763735"
    },
    "ndcg_10": {
        "status": "Completed",
        "duration": "0:00:05.386466",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_3wlostko_20250425_155321_764385"
    },
    "recall_10": {
        "status": "Completed",
        "duration": "0:00:05.389709",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_search_recall_evaluator_searchrecallevaluator_a26a6yyz_20250425_155321_763524"
    }
}


✅ Completed 'hybrid_semantic': {'ndcg_3.NDCG@3': 0.7067437142857143,

[2025-04-25 15:53:52 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:52 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:52 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:53:52 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_ndcg_evaluator_ndcgevaluator_d32c9ug6_20250425_155352_240716, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_d32c9ug6_20250425_155352_240716/logs.txt
[2025-04-25 15:53:52 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_

======= Combined Run Summary (Per Evaluator) =======

{
    "ndcg_3": {
        "status": "Completed",
        "duration": "0:00:04.745410",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_zs1l8n_u_20250425_155338_974086"
    },
    "ndcg_10": {
        "status": "Completed",
        "duration": "0:00:04.741543",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_6ddgmilr_20250425_155338_976526"
    },
    "recall_10": {
        "status": "Completed",
        "duration": "0:00:04.722357",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_search_recall_evaluator_searchrecallevaluator_l8mm4axc_20250425_155338_977975"
    }
}


✅ Completed 'hybrid': {'ndcg_3.NDCG@3': 0.781057142857143, 'ndcg_10.

[2025-04-25 15:54:06 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:54:06 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:54:06 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_evals_custom_ndcg_evaluator_ndcgevaluator_t0m12rtx_20250425_155406_580677, log path: /Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_t0m12rtx_20250425_155406_580677/logs.txt
[2025-04-25 15:54:06 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 15:54:06 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run src_

======= Combined Run Summary (Per Evaluator) =======

{
    "ndcg_3": {
        "status": "Completed",
        "duration": "0:00:04.733636",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_ztosqcby_20250425_155352_239051"
    },
    "ndcg_10": {
        "status": "Completed",
        "duration": "0:00:04.731785",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_ndcg_evaluator_ndcgevaluator_d32c9ug6_20250425_155352_240716"
    },
    "recall_10": {
        "status": "Completed",
        "duration": "0:00:04.731253",
        "completed_lines": 35,
        "failed_lines": 0,
        "log_path": "/Users/marcjimz/.promptflow/.runs/src_evals_custom_search_recall_evaluator_searchrecallevaluator_c4c23yli_20250425_155352_241033"
    }
}


✅ Completed 'keyword': {'ndcg_3.NDCG@3': 0.6384082857142858, 'ndcg_1